In [ ]:
# ============================================================
# Cell 1: Setup environment on L4 runtime
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/cs4782_matcher/Matcher

# Install dependencies
!pip install -q matplotlib torchmetrics torchshow opencv-python timm POT omegaconf iopath tqdm future tensorboardX
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

# Extract FSS-1000 to local disk (faster than reading from Drive)
import os, shutil
FSS_ROOT = '/content/datasets/FSS-1000'
if not os.path.exists(os.path.join(FSS_ROOT, 'data')) or \
   len(os.listdir(os.path.join(FSS_ROOT, 'data'))) < 1000:
    print("Extracting FSS-1000...")
    os.makedirs('/content/datasets', exist_ok=True)
    !unzip -q -o /content/drive/MyDrive/cs4782_matcher/Matcher/datasets/FSS-1000.zip -d /content/datasets/

    data_dir = os.path.join(FSS_ROOT, 'data')
    os.makedirs(data_dir, exist_ok=True)
    cats = [d for d in os.listdir(FSS_ROOT)
            if os.path.isdir(os.path.join(FSS_ROOT, d)) and d not in ('data', 'splits')]
    for cat in cats:
        src = os.path.join(FSS_ROOT, cat)
        dst = os.path.join(data_dir, cat)
        if not os.path.exists(dst):
            shutil.move(src, dst)

    splits_dir = os.path.join(FSS_ROOT, 'splits')
    os.makedirs(splits_dir, exist_ok=True)
    for f in ['test.txt', 'trn.txt', 'val.txt']:
        dst = os.path.join(splits_dir, f)
        if not os.path.exists(dst):
            !wget -q https://raw.githubusercontent.com/juhongm999/hsnet/main/data/splits/fss/{f} -O {dst}

assert len(os.listdir('/content/datasets/FSS-1000/data')) == 1000, "Dataset incomplete"
assert os.path.exists('models/dinov2_vitl14_pretrain.pth'), "DINOv2 missing"
assert os.path.exists('models/sam_vit_h_4b8939.pth'), "SAM missing"

print("\n=== Setup complete ===")
!nvidia-smi | grep -E "L4|T4|MiB" | head -2

In [ ]:
# ============================================================
# Cell 2: Run main FSS-1000 experiment on L4
# Expected runtime: ~3 hours
# ============================================================
%cd /content/drive/MyDrive/cs4782_matcher/Matcher

import os
os.makedirs('/content/drive/MyDrive/cs4782_matcher/results', exist_ok=True)

!stdbuf -oL -eL python -u main_oss.py \
    --benchmark fss \
    --datapath /content/datasets \
    --max_sample_iterations 30 \
    --sample-range "(4,6)" \
    --multimask_output 0 \
    --alpha 0.8 --beta 0.2 --exp 1. \
    --num_merging_mask 10 \
    --fold 0 \
    --log-root "output/fss/fold0_main" 2>&1 | tee /content/drive/MyDrive/cs4782_matcher/results/run_main_live.log

In [ ]:
# ============================================================
# Cell 3: Back up main experiment results
# ============================================================
import os, shutil, glob, json, re

PROJECT_ROOT = '/content/drive/MyDrive/cs4782_matcher'
MATCHER_ROOT = os.path.join(PROJECT_ROOT, 'Matcher')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')

os.makedirs(os.path.join(RESULTS_DIR, 'logs'), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'raw_main_run'), exist_ok=True)

pattern = os.path.join(MATCHER_ROOT, 'output/fss/fold0_main/_TEST_*.log/log.txt')
candidates = glob.glob(pattern)
if not candidates:
    pattern2 = os.path.join(MATCHER_ROOT, 'output/fss/fold0_main/**/log.txt')
    candidates = glob.glob(pattern2, recursive=True)

assert len(candidates) > 0, "ERROR: Could not find main experiment log.txt!"
main_log_src = max(candidates, key=os.path.getmtime)
print(f"Found main log: {main_log_src}")
print(f"Log size: {os.path.getsize(main_log_src)} bytes")

main_log_dst = os.path.join(RESULTS_DIR, 'logs', 'main_with_ilm.log')
shutil.copy(main_log_src, main_log_dst)

live_log = '/content/drive/MyDrive/cs4782_matcher/results/run_main_live.log'
if os.path.exists(live_log):
    shutil.copy(live_log, os.path.join(RESULTS_DIR, 'logs', 'main_with_ilm_live.log'))

log_dir_src = os.path.dirname(main_log_src)
log_dir_dst = os.path.join(RESULTS_DIR, 'raw_main_run')
if os.path.exists(log_dir_dst):
    shutil.rmtree(log_dir_dst)
shutil.copytree(log_dir_src, log_dir_dst)

with open(main_log_dst, 'r') as f:
    text = f.read()
fold_match = re.search(r'Fold\s+\d+\s+mIoU:\s+([\d.]+)\s+FB-IoU:\s+([\d.]+)', text)
assert fold_match, "ERROR: Could not parse final mIoU!"
final_miou   = float(fold_match.group(1))
final_fb_iou = float(fold_match.group(2))

summary = {
    'experiment': 'main (with ILM, alpha=0.8, beta=0.2)',
    'benchmark': 'FSS-1000 fold 0',
    'episodes': 2400,
    'final_mIoU': final_miou,
    'final_FB_IoU': final_fb_iou,
    'paper_reported_mIoU': 87.0,
    'gpu': 'L4',
}
with open(os.path.join(RESULTS_DIR, 'main_experiment_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*60)
print("MAIN EXPERIMENT RESULTS SAFELY BACKED UP")
print("="*60)
print(f"  Final mIoU:   {final_miou:.2f}  (paper: 87.0)")
print(f"  Final FB-IoU: {final_fb_iou:.2f}")

!ls -la /content/drive/MyDrive/cs4782_matcher/results/
!ls -la /content/drive/MyDrive/cs4782_matcher/results/logs/